In [47]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
import pickle
import warnings
warnings.filterwarnings('ignore')

 # STEP 1 : Data Cleaning and Preprocessing

In [52]:
df = pd.read_csv('../data/combined_dns_data.csv')
print(f'\nInitial shape: {df.shape}')
print(f'Label distribution:\n{df["Label"].value_counts()}')


Initial shape: (1167269, 36)
Label distribution:
Label
0    917300
1    249969
Name: count, dtype: int64


In [54]:
# Remove duplicates
initial_rows = df.shape[0]
df = df.drop_duplicates()
duplicate_rows = initial_rows - df.shape[0]
print(f'\nRemoved duplicates: {duplicate_rows} rows')


Removed duplicates: 0 rows


In [55]:
missing_counts = df.isnull().sum()
if missing_counts.sum() > 0:
    df = df.dropna()
    print(f'Removed missing values: {missing_counts.sum()} rows')
else:
    print(' No missing values')

Removed missing values: 16056 rows


# STEP 2: OUTLIER & SKEW TREATMENT

In [61]:
# split features and target
X = df.drop('Label', axis=1)   
y = df['Label']
print(f'\nFeatures shape: {X.shape}, Target shape: {y.shape}')


Features shape: (1159241, 35), Target shape: (1159241,)


In [66]:
#check type of features 
print(f"Feature types:\n{X.dtypes}")

Feature types:
SourceIP                                   object
DestinationIP                              object
SourcePort                                  int64
DestinationPort                             int64
TimeStamp                                  object
Duration                                  float64
FlowBytesSent                               int64
FlowSentRate                              float64
FlowBytesReceived                           int64
FlowReceivedRate                          float64
PacketLengthVariance                      float64
PacketLengthStandardDeviation             float64
PacketLengthMean                          float64
PacketLengthMedian                        float64
PacketLengthMode                            int64
PacketLengthSkewFromMedian                float64
PacketLengthSkewFromMode                  float64
PacketLengthCoefficientofVariation        float64
PacketTimeVariance                        float64
PacketTimeStandardDeviation        

In [64]:

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

In [69]:
# Clip extreme percentiles first (1st and 99th) to handle outliers
for col in numeric_cols:
    q1 = X[col].quantile(0.01)
    q99 = X[col].quantile(0.99)
    X[col] = X[col].clip(lower=q1, upper=q99)

# Apply log1p transform only to positive skewed features
for col in numeric_cols:
    if X[col].min() >= 0:  # Only log1p for non-negative
        skewness = X[col].skew()
        if abs(skewness) > 1.0:  # Highly skewed
            X[col] = np.log1p(X[col])

print(f'Applied percentile clipping and log1p to {len(numeric_cols)} features')

Applied percentile clipping and log1p to 31 features


STEP 3: FEATURE SELECTION & REDUCTION

In [70]:
# Select only numeric features
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
X_numeric = X[numeric_features]

In [74]:

# Drop high-correlation duplicates
corr_matrix = X_numeric.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
X_numeric = X_numeric.drop(columns=to_drop)
print(f'Dropped {len(to_drop)} high-corr features')
print (f'features after dropping: {X_numeric.shape[1]}')


Dropped 0 high-corr features
features after dropping: 26


In [76]:
 # Keep top MI (Mutual Information) features  
mi_scores = mutual_info_classif(X_numeric, y, random_state=42)
mi_df = pd.DataFrame({
    'feature': X_numeric.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False) 
print(f'\nTop 10 features by MI score:\n{mi_df.head(10)}')


Top 10 features by MI score:
                               feature  mi_score
10                    PacketLengthMode  0.570281
9                   PacketLengthMedian  0.482398
2                        FlowBytesSent  0.408122
8                     PacketLengthMean  0.406707
7        PacketLengthStandardDeviation  0.388389
11          PacketLengthSkewFromMedian  0.335920
13  PacketLengthCoefficientofVariation  0.329930
1                             Duration  0.313055
12            PacketLengthSkewFromMode  0.312615
14                    PacketTimeMedian  0.210809


In [77]:
# Keep top MI (Mutual Information) features  
mi_scores = mutual_info_classif(X_numeric, y, random_state=42)
mi_df = pd.DataFrame({
    'feature': X_numeric.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

# Keep top 20 features (or all if less than 20)
top_k = min(20, len(mi_df))
top_features = mi_df.head(top_k)['feature'].tolist()
X = X_numeric[top_features]

print(f'Selected top {top_k} MI features from {len(mi_df)} total')
print(f'Final feature set: {X.shape[1]} features')

Selected top 20 MI features from 26 total
Final feature set: 20 features


In [78]:
print('\n' + '=' * 60)
print('STEP 6: TRAIN / VAL / TEST SPLIT (70/15/15)')
print('=' * 60)

# First split: 70% train, 30% temp (val+test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Second split: Split temp into 50% val, 50% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f'\nTrain: {X_train.shape[0]} rows ({100*len(X_train)/len(X):.0f}%)')
print(f'Val:   {X_val.shape[0]} rows ({100*len(X_val)/len(X):.0f}%)')
print(f'Test:  {X_test.shape[0]} rows ({100*len(X_test)/len(X):.0f}%)')

print(f'\nTrain class balance: {y_train.value_counts()[0]} benign, {y_train.value_counts()[1]} malicious')
print(f'Val class balance:   {y_val.value_counts()[0]} benign, {y_val.value_counts()[1]} malicious')
print(f'Test class balance:  {y_test.value_counts()[0]} benign, {y_test.value_counts()[1]} malicious')


STEP 6: TRAIN / VAL / TEST SPLIT (70/15/15)

Train: 811468 rows (70%)
Val:   173886 rows (15%)
Test:  173887 rows (15%)

Train class balance: 636688 benign, 174780 malicious
Val class balance:   136433 benign, 37453 malicious
Test class balance:  136434 benign, 37453 malicious


In [79]:
print('\n' + '=' * 60)
print('STEP 4: FEATURE SCALING (RobustScaler)')
print('=' * 60)

# RobustScaler uses IQR (Interquartile Range) - resistant to outliers
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print(f'Scaled train - median: {X_train_scaled.values.mean():.4f}, IQR preserved')
print('RobustScaler fit on train set only (no data leakage)')


STEP 4: FEATURE SCALING (RobustScaler)
Scaled train - median: 0.2158, IQR preserved
RobustScaler fit on train set only (no data leakage)


In [80]:
print('\n' + '=' * 60)
print('STEP 5: CLASS IMBALANCE HANDLING (SMOTE)')
print('=' * 60)

print('\nBefore SMOTE:')
print(f'  Benign: {(y_train == 0).sum():,}')
print(f'  Malicious: {(y_train == 1).sum():,}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print('\nAfter SMOTE:')
print(f'  Benign: {(y_train_balanced == 0).sum():,}')
print(f'  Malicious: {(y_train_balanced == 1).sum():,}')

X_train_balanced = pd.DataFrame(X_train_balanced, columns=X.columns)


STEP 5: CLASS IMBALANCE HANDLING (SMOTE)

Before SMOTE:
  Benign: 636,688
  Malicious: 174,780

After SMOTE:
  Benign: 636,688
  Malicious: 636,688


In [83]:

print('PREPROCESSING VERIFICATION')

print(f'\nTrain (SMOTE balanced): {X_train_balanced.shape}')
print(f'Val (original): {X_val_scaled.shape}')
print(f'Test (original): {X_test_scaled.shape}')
print(f'\nNull values in train: {X_train_balanced.isnull().sum().sum()}')
print(f'Null values in val: {X_val_scaled.isnull().sum().sum()}')
print(f'Null values in test: {X_test_scaled.isnull().sum().sum()}')


PREPROCESSING VERIFICATION

Train (SMOTE balanced): (1273376, 20)
Val (original): (173886, 20)
Test (original): (173887, 20)

Null values in train: 0
Null values in val: 0
Null values in test: 0


In [84]:

print('SAVING PREPROCESSED DATA')

train_df = X_train_balanced.copy()
train_df['Label'] = y_train_balanced.values

val_df = X_val_scaled.copy()
val_df['Label'] = y_val.values

test_df = X_test_scaled.copy()
test_df['Label'] = y_test.values

train_output = Path('../data/train_data.csv')
val_output = Path('../data/val_data.csv')
test_output = Path('../data/test_data.csv')

train_df.to_csv(train_output, index=False)
val_df.to_csv(val_output, index=False)
test_df.to_csv(test_output, index=False)

print(f'✓ Train data: {train_df.shape[0]:,} rows → {train_output.name}')
print(f'✓ Val data:   {val_df.shape[0]:,} rows → {val_output.name}')
print(f'✓ Test data:  {test_df.shape[0]:,} rows → {test_output.name}')

SAVING PREPROCESSED DATA
✓ Train data: 1,273,376 rows → train_data.csv
✓ Val data:   173,886 rows → val_data.csv
✓ Test data:  173,887 rows → test_data.csv


In [85]:

print('SAVING PREPROCESSING METADATA')


# Save scaler
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('✓ RobustScaler saved')

# Save feature names
with open('../data/feature_names.txt', 'w') as f:
    f.write('\n'.join(X.columns.tolist()))
print(f'✓ Feature names saved ({len(X.columns)} features)')

# Save preprocessing config
config_text = f"""Preprocessing Configuration
===========================
Data Cleaning: Removed duplicates, NaN, inf values
Outlier Treatment: Log1p transform + 1st/99th percentile clipping
Feature Selection: Dropped high-corr (>0.95), kept top 20 MI features
Feature Scaling: RobustScaler (IQR-based, fit on train only)
Class Balancing: SMOTE on training set
Train/Val/Test Split: 70% / 15% / 15%

Final Dataset:
  - Train (balanced): {len(train_df):,} rows
  - Val (original): {len(val_df):,} rows
  - Test (original): {len(test_df):,} rows
  - Features: {X.shape[1]}
"""

with open('../data/preprocessing_config.txt', 'w') as f:
    f.write(config_text)
print('✓ Preprocessing config saved')

print(f'\n✓ Ready for model training ({X.shape[1]} features)')

SAVING PREPROCESSING METADATA
✓ RobustScaler saved
✓ Feature names saved (20 features)
✓ Preprocessing config saved

✓ Ready for model training (20 features)
